# TG-119: Conventional Code-Based Planning

![gallery_thumbnail](_static/notebooks/code_tg119_conv/cover.png)

## Intro

Welcome to the <i>TG-119: Conventional Code-Based Planning</i> notebook! <br><br> In this notebook, we will showcase a beginner-friendly version of the package's code-based functionality using data from the TG-119 standard case (available from our [Github repository's docs folder](https://github.com/pyanno4rt/pyanno4rt/tree/master/docs) as .mat-files) for conventional treatment planning.

## Import of the relevant classes

In [ ]:
from pyanno4rt.base import (
    Configuration, Evaluation, Optimization, TreatmentPlan)

Working with the command line interface (CLI) or an interactive development environment (IDE) requires you to initialize an object of the class `pyanno4rt.base._treatment_plan.TreatmentPlan`. It takes three subobjects structuring the parameter space as arguments:

> `pyanno4rt.base._configuration.Configuration`<br>
> <i>handles design parameters w.r.t. general plan configuration and external data links</i>

> `pyanno4rt.base._optimization.Optimization`<br>
> <i>handles design parameters w.r.t. the optimization components and settings</i>

> `pyanno4rt.base._evaluation.Evaluation`<br>
> <i>handles design parameters w.r.t. the treatment plan evaluation methods</i>

## Treatment plan initialization

Initializing an object from `pyanno4rt.base._treatment_plan.TreatmentPlan` includes setting up the subobjects. For the sake of readability and instructiveness, we will define these one by one.

### Setting up the configuration object

In [ ]:
configuration = Configuration(
    label='TG-119-conv',  # Unique identifier for the treatment plan
    modality='photon',  # Treatment modality
    imaging_path='./TG_119_data.mat',  # Path to the CT and segmentation data
    dose_matrix_path='./TG_119_photonDij.mat',  # Path to the dose-influence matrix
    dose_resolution=[6, 6, 6],  # Size of the dose grid in [mm] per dimension
    min_log_level='info',  # Minimum logging level
    number_of_fractions=30  # Number of fractions
    )

We decide to label our plan 'TG-119-conv' and set the modality to 'photon'. Since we have the MATLAB files available for the TG-119 case, we provide the corresponding paths to the imaging and dose-influence matrix files (you may adapt them). Also, knowing that the dose-influence matrix has been calculated with a resolution of 6x6x6 mm<sup>3</sup>, the dose resolution parameter can be specified accordingly. Finally, we stick to the default values 'info' for the minimum logging level, which means that any debugging messages will be suppressed, and 30 for the number of fractions.

### Setting up the optimization object

In [ ]:
from pyanno4rt.optimization.components import (
    SquaredDeviation, SquaredOverdosing)

optimization = Optimization(
    components=[  # Optimization components for each segment of interest
        SquaredOverdosing(  # Quadratic penalty for excess dose to the core
            segment='Core', maximum_dose=25, weight=100),
        SquaredDeviation(  # Quadratic penalty for deviating dose to the outer target
            segment='OuterTarget', target_dose=60, weight=1000),
        SquaredOverdosing(  # Quadratic penalty for excess dose to the body
            segment='BODY', maximum_dose=30, weight=800)],
    method='weighted-sum',  # Single- or multi-criteria optimization method
    solver='scipy',  # Python package to be used for solving the optimization problem
    algorithm='L-BFGS-B',  # Solution algorithm from the chosen solver
    initial_strategy='target-coverage',  # Initialization strategy for the fluence vector
    initial_fluence_vector=None,  # User-defined initial fluence vector (only for 'warm-start')
    lower_variable_bounds=0,  # Lower bounds on the decision variables
    upper_variable_bounds=None,  # Upper bounds on the decision variables
    maximum_iterations=500,  # Maximum number of iterations for the solvers to converge
    tolerance=0.001  # Precision goal for the objective function value
    )

Next, we need to describe how the TG-119 treatment plan should be optimized. In general, the final plan should apply a reasonably high dose to the target volume while limiting the dose exposure to relevant organs at risk to prevent post-treatment complications. <br> To achieve this, we import and initialize optimization components for the core ('Core'), for the outer target ('OuterTarget'), and for the whole body ('BODY'), from the classes:

> `pyanno4rt.optimization.components._squared_overdosing.SquaredOverdosing`<br>
> <i>refers to a function that penalizes values above a maximum dose</i>

> `pyanno4rt.optimization.components._squared_deviation.SquaredDeviation`<br>
> <i>refers to a function that penalizes bilateral deviations from a target dose</i>

Please check out the <i>Optimization components</i> notebook in the [Notebooks](https://pyanno4rt.readthedocs.io/en/latest/notebooks.html) section for more information on the components.

Once the component list has been defined, we find ourselves in a trade-off situation, where a higher degree of fulfillment for one objective is usually accompanied with a lower degree of fulfillment for another. We can handle this by choosing the 'weighted-sum' method, which scalarizes the multi-objective problem by multiplying each objective value with a weight parameter and then summing them up. This works well with the default solution algorithm, the 'L-BFGS-B' algorithm from the 'scipy' solver, so we pick that one. For the initialization of the fluence vector (holding the decision variables), we opt for 'target-coverage' to start off with a satisfactory dose level for the outer target. We put a lower bound of 0 and no upper bound (None) on the fluence, matching its physical properties. As the final step, we limit the number of iterations to 500 and the tolerance (precision goal) for the objective function value to 0.001.

### Setting up the evaluation object

In [ ]:
evaluation = Evaluation(
    dvh_type='cumulative',  # Type of DVH to be calculated
    number_of_points=1000,  # Number of (evenly-spaced) points for which to evaluate the DVH
    reference_volume=[2, 5, 50, 95, 98],  # Reference volumes for which to calculate the inverse DVH values
    reference_dose=[],  # Reference dose values for which to calculate the DVH values
    display_segments=[],  # Names of the segmented structures to be displayed
    display_metrics=[]  # Names of the plan evaluation metrics to be displayed
    )

It is not actually necessary to pass any evaluation parameters if you are happy with the default values. However, we fully initialize the evaluation object for reasons of completeness. First, we select the DVH type 'cumulative' and request its evaluation at 1000 (evenly-spaced) points. With the parameters 'reference_volume' and 'reference_dose', we let the package calculate dose and volume quantiles at certain levels (by inserting an empty list for 'reference_dose', the levels are automatically determined). The last two parameters, 'display_segments' and 'display_metrics', can be used to filter the segments and metrics to be displayed later in the treatment plan visualization. We also specify empty lists here to not exclude any segment or metric.

### Extracting the parameter sets as dictionaries

If you need to view the treatment plan parameters in a compact form, each subobject provides a `to_dict` method, which returns a parameter dictionary.

In [ ]:
config_dict = configuration.to_dict()
opt_dict = optimization.to_dict()
eval_dict = evaluation.to_dict()

### Initializing the base class

In [ ]:
tp = TreatmentPlan(configuration, optimization, evaluation)

## Treatment plan workflow

The code-based interface equips each treatment plan with methods for configuration, optimization, evaluation and visualization, all of which can be called parameter-free. These workflow steps will be described step-by-step in the following.

### Configuring the plan

In [ ]:
tp.configure()

First, a successfully initialized treatment plan needs to be configured. By calling the `configure` method, the information from the configuration object is passed to internal classes, which perform functional (e.g. logging) or I/O tasks (e.g. image data processing).

### Modeling for the plan

In [ ]:
tp.model()

If any machine learning model-based components are present, they must be set up in the second step. Therefore, each treatment plan has a `model` method to load/fit the prediction models, optionally evaluate and inspect them, and integrate them with the respective optimization components. Here, no machine learning components exist, which means that this method can also be skipped. Check out the "TG-119: Code-Based Planning with Machine Learning Outcome Models" notebook to learn more about machine learning outcome model-based optimization in pyanno4rt.

### Optimizing the plan

In [ ]:
tp.optimize()

Afterwards, the treatment plan is ready for optimization. We call the `optimize` method, which generates the internal optimization classes by passing the parameters from the optimization object, and at the end triggers the solver run.

### Evaluating the plan

In [ ]:
tp.evaluate()

The penultimate step is the evaluation of the treatment plan, and following the previous logic, we added an `evaluate` method. Internally, this creates objects from the DVH and dosimetrics classes, which take the parameters from the evaluation object and run the evaluation methods.

### Visualizing the plan

In [ ]:
tp.visualize()

To complete the standard workflow, we can analyze the results of the treatment plan optimization and evaluation both qualitatively and quantitatively. Our package features a visual analysis tool that provides three sets of visualizations: optimization problem analysis, data-driven model review, and treatment plan evaluation. It can easily be launched by the `visualize` method.

![pyanno4rt visualizer](_static/notebooks/code_tg119_conv/component_graph.png)

### Shortcut: composing the plan

In [ ]:
tp.compose()

Many times you will just run all five of the above methods in sequence. To make this a little more convenient, the treatment plan can also be "composed" in a single step, using the appropriately named `compose` method (and yeah, we love music ❤️).

### Updating parameter values

In [ ]:
tp.update({
    'modality': 'photon',
    'dvh_type': 'cumulative'
    })

One last class functionality is the updating of parameter values with the `update` method. This comes in handy because each of the `configure`, `model`/`optimize` and `evaluate` methods is based on a single subobject, so that, for example, the `evaluate` method can be called again after updating an evaluation parameter without repeating the prior workflow steps. <br>
The `update` method takes a dictionary with key-value pairs as input, where the former are from the input parameter names, and the latter are the new parameter values. We do not want to change the plan at this point, so we will just overwrite the modality and the DVH type with the previous values for illustration purposes.

### Saving and loading treatment plans

In [ ]:
from pyanno4rt.tools import copycat, snapshot

# Save the treatment plan to a snapshot
snapshot(
    instance=tp, path='./',
    include_patient_data=False, include_dose_matrix=False,
    include_model_data=False, include_optimum=False)

# Load the treatment plan as a copycat
tp_copy = copycat(base_class=TreatmentPlan, path='./TG-119-example/')

Treatment plans generated with *pyanno4rt* can be saved with the `snapshot` and loaded with the `copycat` function.<br>
A snapshot folder includes a JSON file with the parameter dictionaries, a compiled log file, and, if machine learning model-based components are used, subfolders with model configuration files. Optionally, you can specify whether to add the imaging data, the dose-influence matrix, the model training data and the optimized fluence vector (this allows sharing a treatment plan with all data). The name of the snapshot folder is specified by the treatment plan label from the configuration object.<br>
Conversely, a snapshot that has been saved can be loaded back into a Python variable as a copycat with the base class to be initialized and the folder path as inputs.

## Outro

We hope that this little example illustrates the basic usage of the code-based *pyanno4rt* interface for conventional treatment planning. If you have any remarks, please take a look at the [Help and Support](https://pyanno4rt.readthedocs.io/en/latest/help_support.html) section and drop us a line. We would also be happy if you leave a positive comment and recommend our work to others. <br><br> Thank you for using *pyanno4rt* 😊